# Pelanca NNUE v3

- Hidden 16, Batch 64K, MSE puro, sem material anchor

In [ ]:
!pip install python-chess -q 2>/dev/null
!pip install h5py tqdm -q 2>/dev/null

import chess
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import h5py
import struct
import os
import time
from tqdm.auto import tqdm
from numba import njit, prange

print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
MIN_DEPTH = 18
MAX_ABS_CP = 3000
EVAL_SCALE = 400.0
HDF5_FILE = '/kaggle/working/positions_v2.h5'
RAW_DIR = '/kaggle/working/parquets'
os.makedirs(RAW_DIR, exist_ok=True)

PARQUET_IDS = ['00015', '00016']
BASE_URL = 'https://huggingface.co/datasets/Lichess/chess-position-evaluations/resolve/main/data'

INPUT_SIZE = 768
FT_SIZE = 256
HIDDEN_SIZE = 16

EPOCHS = 80
BATCH_SIZE = 65536
LR_MAX = 1e-3
LR_MIN = 1e-5
WEIGHT_DECAY = 1e-6
MIRROR_AUG = True

FT_QUANT = 64
HIDDEN_QUANT = 64
OUTPUT_QUANT = 64

NNUE_OUTPUT = '/kaggle/working/pelanca_v3.nnue'
CKPT_DIR = '/kaggle/working/checkpoints_v3'
os.makedirs(CKPT_DIR, exist_ok=True)

params = INPUT_SIZE*FT_SIZE + FT_SIZE + FT_SIZE*2*HIDDEN_SIZE + HIDDEN_SIZE + HIDDEN_SIZE + 1
print(f'Arch: {INPUT_SIZE}->{FT_SIZE}->{HIDDEN_SIZE}->1 ({params:,} params)')
print(f'Batch: {BATCH_SIZE:,} | Epochs: {EPOCHS}')

---
## Download Direto (wget ~2 min) + Filtro com Pandas

In [ ]:
if os.path.exists(HDF5_FILE):
    size_mb = os.path.getsize(HDF5_FILE) / 1024**2
    print(f'Dataset existe: {HDF5_FILE} ({size_mb:.0f} MB)')
else:
    t0 = time.time()

    # === 1. DOWNLOAD silencioso ===
    for pid in PARQUET_IDS:
        fname = f'train-{pid}-of-00017.parquet'
        fpath = os.path.join(RAW_DIR, fname)
        if os.path.exists(fpath):
            sz = os.path.getsize(fpath) / 1024**2
            if sz > 100:  # arquivo valido (>100MB)
                print(f'  {fname} ja existe ({sz:.0f} MB)')
                continue
            else:
                os.remove(fpath)  # corrompido, re-baixar
        url = f'{BASE_URL}/{fname}?download=true'
        print(f'  Baixando {fname}...', end=' ', flush=True)
        ret = os.system(f'wget -q -O "{fpath}" "{url}" 2>/dev/null')
        sz = os.path.getsize(fpath) / 1024**2 if os.path.exists(fpath) else 0
        print(f'{sz:.0f} MB' if ret == 0 else 'ERRO!')

    dl_time = time.time() - t0
    print(f'\nDownload: {dl_time/60:.1f} min\n')

    # === 2. FILTRAR 100% VECTORIZADO (sem iterrows, sem chess.Board) ===
    # ~2 min para 50M posicoes em vez de 2h
    all_fens = []
    all_evals = []
    total_raw = 0

    for pid in PARQUET_IDS:
        fname = f'train-{pid}-of-00017.parquet'
        fpath = os.path.join(RAW_DIR, fname)
        print(f'Processando {fname}...')
        t1 = time.time()

        df = pd.read_parquet(fpath, columns=['fen', 'depth', 'cp', 'mate'])
        total_raw += len(df)
        print(f'  Bruto: {len(df):,}')

        # Filtro depth (vectorizado)
        df = df[df['depth'] >= MIN_DEPTH]

        # Filtro cp extremo (vectorizado)
        mask_cp = df['cp'].notna() & (df['cp'].abs() <= MAX_ABS_CP)
        mask_mate = df['mate'].notna() & (df['mate'].abs() > 3)
        df = df[mask_cp | mask_mate]

        # Dedup (vectorizado)
        df = df.drop_duplicates(subset='fen')

        # Calcular eval (vectorizado - sem loop!)
        evals = np.zeros(len(df), dtype=np.float32)
        cp_mask = df['cp'].notna().values
        mate_mask = df['mate'].notna().values
        cp_vals = df['cp'].fillna(0).values.astype(np.float64)
        mate_vals = df['mate'].fillna(0).values.astype(np.float64)

        evals[cp_mask] = np.tanh(cp_vals[cp_mask] / EVAL_SCALE).astype(np.float32)
        evals[mate_mask & (mate_vals > 0)] = 1.0
        evals[mate_mask & (mate_vals < 0)] = -1.0

        # Validar FEN por string (sem chess.Board!)
        # FEN valido: tem exatamente 1 rei branco e 1 preto, 6 partes
        fens = df['fen'].values
        valid = np.ones(len(fens), dtype=bool)
        for i in range(len(fens)):
            fen = fens[i]
            board_part = fen.split(' ')[0] if ' ' in fen else fen
            # Deve ter exatamente 1 K e 1 k
            if board_part.count('K') != 1 or board_part.count('k') != 1:
                valid[i] = False

        fens = fens[valid]
        evals = evals[valid]

        all_fens.extend(fens.tolist())
        all_evals.extend(evals.tolist())

        elapsed = time.time() - t1
        print(f'  Filtrado: {len(fens):,} posicoes em {elapsed:.0f}s')

        del df, fens, evals, valid
        import gc; gc.collect()

    # Dedup global (entre parquets)
    print(f'\nDedup global...')
    seen = set()
    unique_fens = []
    unique_evals = []
    for f, e in zip(all_fens, all_evals):
        if f not in seen:
            seen.add(f)
            unique_fens.append(f)
            unique_evals.append(e)

    total_kept = len(unique_fens)
    print(f'Total unico: {total_kept:,} de {total_raw:,} ({total_kept/total_raw*100:.0f}%)')

    del all_fens, all_evals, seen
    gc.collect()

    # === 3. SHUFFLE + SPLIT + SALVAR HDF5 ===
    rng = np.random.default_rng(42)
    indices = rng.permutation(total_kept)
    val_n = int(total_kept * 0.03)
    val_idx, train_idx = indices[:val_n], indices[val_n:]

    fens_arr = np.array(unique_fens, dtype=object)
    evals_arr = np.array(unique_evals, dtype=np.float32)
    # WDL = eval como proxy (dataset nao tem resultado do jogo)
    wdl_arr = evals_arr.copy()

    dt_str = h5py.string_dtype()
    with h5py.File(HDF5_FILE, 'w') as f:
        for name, idx in [('train', train_idx), ('val', val_idx)]:
            g = f.create_group(name)
            g.create_dataset('fens', data=fens_arr[idx], dtype=dt_str)
            g.create_dataset('evals', data=evals_arr[idx], compression='gzip')
            g.create_dataset('wdl', data=wdl_arr[idx], compression='gzip')
        f.attrs['total'] = total_kept
        f.attrs['min_depth'] = MIN_DEPTH

    elapsed = time.time() - t0
    size_mb = os.path.getsize(HDF5_FILE) / 1024**2
    print(f'\nSalvo: {HDF5_FILE} ({size_mb:.0f} MB) em {elapsed/60:.1f} min')
    print(f'  Train: {len(train_idx):,} | Val: {len(val_idx):,}')

    # Limpar parquets
    for pid in PARQUET_IDS:
        p = os.path.join(RAW_DIR, f'train-{pid}-of-00017.parquet')
        if os.path.exists(p): os.remove(p)
    print('Parquets removidos')

    del unique_fens, unique_evals, fens_arr, evals_arr, wdl_arr
    import gc; gc.collect()

---
## Pre-processar FENs -> arrays binarios (1x) + Dataset (instantaneo)

In [ ]:
_PIECE_MAP = {
    'P': (0, 0), 'N': (0, 1), 'B': (0, 2), 'R': (0, 3), 'Q': (0, 4), 'K': (0, 5),
    'p': (1, 0), 'n': (1, 1), 'b': (1, 2), 'r': (1, 3), 'q': (1, 4), 'k': (1, 5),
}

def fen_to_sparse_fast(fen_str):
    if isinstance(fen_str, bytes): fen_str = fen_str.decode()
    parts = fen_str.split(' ')
    stm = 0 if parts[1] == 'w' else 1
    wi, bi = [], []
    sq = 56
    for ch in parts[0]:
        if ch == '/': sq -= 16
        elif ch.isdigit(): sq += int(ch)
        else:
            color, pt = _PIECE_MAP[ch]
            wi.append(color * 384 + pt * 64 + sq)
            bi.append((1 - color) * 384 + pt * 64 + (sq ^ 56))
            sq += 1
    return stm, wi, bi


PREPROCESSED_FILE = HDF5_FILE.replace('.h5', '_preprocessed.h5')

def preprocess_split(h5_src, h5_dst, split, max_pos=None):
    import gc
    with h5py.File(h5_src, 'r') as f:
        total = len(f[split]['evals'])
        n = min(total, max_pos) if max_pos else total
        evals = f[split]['evals'][:n].astype(np.float32)
    print(f'[{split}] Pre-processando {n:,} FENs...')
    max_pieces = n * 32
    w_flat = np.empty(max_pieces, dtype=np.int16)
    b_flat = np.empty(max_pieces, dtype=np.int16)
    offsets = np.empty(n + 1, dtype=np.int32)
    stm = np.empty(n, dtype=np.uint8)
    CHUNK = 500_000
    pos = 0
    offsets[0] = 0
    for cs in tqdm(range(0, n, CHUNK), desc=f'  {split}', total=(n+CHUNK-1)//CHUNK):
        ce = min(cs + CHUNK, n)
        with h5py.File(h5_src, 'r') as f:
            fc = f[split]['fens'][cs:ce]
        for j, rf in enumerate(fc):
            i = cs + j
            s, wi, bi = fen_to_sparse_fast(rf)
            k = len(wi)
            w_flat[pos:pos+k] = wi
            b_flat[pos:pos+k] = bi
            pos += k
            offsets[i+1] = pos
            stm[i] = s
        del fc
        gc.collect()
    w_flat = w_flat[:pos]
    b_flat = b_flat[:pos]
    with h5py.File(h5_dst, 'a') as f:
        g = f.require_group(split)
        for name, arr in [('w_flat', w_flat), ('b_flat', b_flat),
                          ('offsets', offsets), ('stm', stm), ('evals', evals)]:
            if name in g: del g[name]
            g.create_dataset(name, data=arr)
    mb = (w_flat.nbytes+b_flat.nbytes+offsets.nbytes+evals.nbytes+stm.nbytes)/1e6
    print(f'  Salvo: {mb:.0f} MB')
    del w_flat, b_flat, offsets, stm, evals
    gc.collect()

if os.path.exists(PREPROCESSED_FILE):
    with h5py.File(PREPROCESSED_FILE, 'r') as f:
        if 'train' in f and 'w_flat' in f['train']:
            print(f'Pre-processado existe: {PREPROCESSED_FILE}')
        else:
            preprocess_split(HDF5_FILE, PREPROCESSED_FILE, 'train')
            preprocess_split(HDF5_FILE, PREPROCESSED_FILE, 'val', max_pos=1_500_000)
else:
    preprocess_split(HDF5_FILE, PREPROCESSED_FILE, 'train')
    preprocess_split(HDF5_FILE, PREPROCESSED_FILE, 'val', max_pos=1_500_000)


class NnueSparseDatasetV3(Dataset):
    def __init__(self, h5_path, split='train', mirror=False):
        self.mirror = mirror
        t0 = time.time()
        with h5py.File(h5_path, 'r') as f:
            g = f[split]
            self.w_flat = g['w_flat'][:]
            self.b_flat = g['b_flat'][:]
            self.offsets = g['offsets'][:].astype(np.int64)
            self.stm = g['stm'][:]
            self.evals = g['evals'][:]
        self.n = len(self.evals)
        elapsed = time.time() - t0
        mb = (self.w_flat.nbytes+self.b_flat.nbytes+self.offsets.nbytes+self.evals.nbytes+self.stm.nbytes)/1e6
        print(f'[{split}] {self.n:,} em {elapsed:.1f}s ({mb:.0f} MB)' + (f' (x2={self.n*2:,})' if mirror else ''))

    def __len__(self): return self.n * 2 if self.mirror else self.n


# ============================================================
# NUMBA: batch builder compilado para codigo nativo
# Zero np.repeat, 1 loop paralelo, ~10x mais rapido que numpy
# ============================================================

@njit(cache=True)
def _batch_offsets(offsets_arr, batch_idx, n, mirror):
    B = len(batch_idx)
    lengths = np.empty(B, dtype=np.int64)
    batch_off = np.empty(B, dtype=np.int64)
    for i in range(B):
        idx = batch_idx[i]
        ri = idx - n if (mirror and idx >= n) else idx
        lengths[i] = offsets_arr[ri + 1] - offsets_arr[ri]
    total = np.int64(0)
    for i in range(B):
        batch_off[i] = total
        total += lengths[i]
    return lengths, batch_off, total


@njit(parallel=True, cache=True)
def _fill_batch(w_flat, b_flat, offsets_arr, stm, evals,
                batch_idx, n, mirror,
                lengths, batch_off,
                out_w, out_b, out_ev):
    B = len(batch_idx)
    for i in prange(B):
        idx = batch_idx[i]
        is_mirror = mirror and (idx >= n)
        ri = idx - n if is_mirror else idx
        start = offsets_arr[ri]
        out_pos = batch_off[i]
        length = lengths[i]
        is_black = stm[ri] == 1
        for j in range(length):
            w = np.int64(w_flat[start + j])
            b = np.int64(b_flat[start + j])
            if is_mirror:
                w ^= 7
                b ^= 7
            if is_black:
                w, b = b, w
            out_w[out_pos + j] = w
            out_b[out_pos + j] = b
        ev = evals[ri]
        if is_black:
            ev = -ev
        out_ev[i] = ev


def make_batches(ds, batch_size, shuffle=True):
    n = len(ds)
    perm = np.random.permutation(n).astype(np.int64) if shuffle else np.arange(n, dtype=np.int64)
    max_total = batch_size * 32
    # Double buffering: 2 buffers alternados, elimina .copy() (224MB/batch)
    bufs = [
        (np.empty(max_total, dtype=np.int64), np.empty(max_total, dtype=np.int64), np.empty(batch_size, dtype=np.float32)),
        (np.empty(max_total, dtype=np.int64), np.empty(max_total, dtype=np.int64), np.empty(batch_size, dtype=np.float32)),
    ]
    mirror = ds.mirror
    buf_idx = 0
    for bs in range(0, n - batch_size + 1, batch_size):
        batch_idx = perm[bs:bs + batch_size]
        B = len(batch_idx)
        out_w, out_b, out_ev = bufs[buf_idx]
        lengths, batch_off, total = _batch_offsets(ds.offsets, batch_idx, np.int64(ds.n), mirror)
        if total > len(out_w):
            out_w = np.empty(total, dtype=np.int64)
            out_b = np.empty(total, dtype=np.int64)
            bufs[buf_idx] = (out_w, out_b, out_ev)
        _fill_batch(ds.w_flat, ds.b_flat, ds.offsets, ds.stm, ds.evals,
                    batch_idx, np.int64(ds.n), mirror,
                    lengths, batch_off, out_w, out_b, out_ev)
        yield (
            torch.from_numpy(out_w[:total]),
            torch.from_numpy(batch_off),
            torch.from_numpy(out_b[:total]),
            torch.from_numpy(batch_off.copy()),
            torch.from_numpy(out_ev[:B]),
        )
        buf_idx = 1 - buf_idx


# Warmup numba (JIT compila na primeira chamada, ~5s)
print('Compilando numba...', end=' ', flush=True)
_dummy_off = np.array([0, 2, 5], dtype=np.int64)
_dummy_idx = np.array([0, 1], dtype=np.int64)
_dummy_w = np.array([1,2,3,4,5], dtype=np.int16)
_dummy_b = np.array([6,7,8,9,10], dtype=np.int16)
_dummy_stm = np.array([0, 1], dtype=np.uint8)
_dummy_ev = np.array([0.5, -0.3], dtype=np.float32)
_l, _bo, _t = _batch_offsets(_dummy_off, _dummy_idx, np.int64(2), False)
_ow = np.empty(5, dtype=np.int64)
_ob = np.empty(5, dtype=np.int64)
_oe = np.empty(2, dtype=np.float32)
_fill_batch(_dummy_w, _dummy_b, _dummy_off, _dummy_stm, _dummy_ev,
            _dummy_idx, np.int64(2), False, _l, _bo, _ow, _ob, _oe)
print('OK')

---
## Modelo NNUE v2 (768â†’256â†’8â†’1)

In [ ]:
class ClippedReLU(nn.Module):
    def forward(self, x): return torch.clamp(x, 0.0, 1.0)

class PelancaNNUEv3(nn.Module):
    def __init__(self):
        super().__init__()
        self.ft = nn.EmbeddingBag(INPUT_SIZE, FT_SIZE, mode='sum', sparse=False)
        self.ft_bias = nn.Parameter(torch.zeros(FT_SIZE))
        self.hidden = nn.Linear(FT_SIZE * 2, HIDDEN_SIZE)
        self.out = nn.Linear(HIDDEN_SIZE, 1)
        self.crelu = ClippedReLU()
        self._init()

    def _init(self):
        nn.init.kaiming_normal_(self.ft.weight, nonlinearity='relu')
        nn.init.zeros_(self.ft_bias)
        nn.init.kaiming_normal_(self.hidden.weight, nonlinearity='relu')
        nn.init.zeros_(self.hidden.bias)
        nn.init.xavier_normal_(self.out.weight)
        nn.init.zeros_(self.out.bias)

    def forward(self, stm_idx, stm_off, nstm_idx, nstm_off):
        with torch.amp.autocast('cuda', enabled=False):
            stm_acc = self.crelu(self.ft(stm_idx, stm_off) + self.ft_bias)
            nstm_acc = self.crelu(self.ft(nstm_idx, nstm_off) + self.ft_bias)
        h = self.crelu(self.hidden(torch.cat([stm_acc, nstm_acc], dim=1)))
        return self.out(h)

def nnue_loss(pred, target):
    return F.mse_loss(torch.tanh(pred.float()), target)

m = PelancaNNUEv3()
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
del m

---
## Export (formato binario v2)

In [ ]:
def export_nnue_v3(model, path, verbose=True):
    model.eval()
    m = model.module if hasattr(model, 'module') else model
    with open(path, 'wb') as f:
        f.write(b'PLNN')
        f.write(struct.pack('<I', 3))
        f.write(struct.pack('<I', INPUT_SIZE))
        f.write(struct.pack('<I', FT_SIZE))
        f.write(struct.pack('<I', HIDDEN_SIZE))
        ft_w = (m.ft.weight.data.cpu().T * FT_QUANT).round().clamp(-32767, 32767).to(torch.int16)
        ft_b = (m.ft_bias.data.cpu() * FT_QUANT).round().clamp(-32767, 32767).to(torch.int16)
        f.write(ft_w.contiguous().numpy().tobytes())
        f.write(ft_b.numpy().tobytes())
        h_w = (m.hidden.weight.data.cpu() * HIDDEN_QUANT).round().clamp(-127, 127).to(torch.int8)
        h_b = (m.hidden.bias.data.cpu() * FT_QUANT * HIDDEN_QUANT).round().clamp(-2**30, 2**30).to(torch.int32)
        f.write(h_w.numpy().tobytes())
        f.write(h_b.numpy().tobytes())
        o_w = (m.out.weight.data.cpu() * OUTPUT_QUANT).round().clamp(-127, 127).to(torch.int8)
        o_b = (m.out.bias.data.cpu() * FT_QUANT * HIDDEN_QUANT * OUTPUT_QUANT).round().clamp(-2**30, 2**30).to(torch.int32)
        f.write(o_w.numpy().tobytes())
        f.write(o_b.numpy().tobytes())
    if verbose:
        sz = os.path.getsize(path)
        print(f'Exportado: {path} ({sz:,} bytes)')

print('Export v3 OK')

---
## Carregar Dataset

In [ ]:
import gc; gc.collect()
print('Carregando arrays pre-processados...\n')

train_ds = NnueSparseDatasetV3(PREPROCESSED_FILE, 'train', mirror=MIRROR_AUG)
val_ds = NnueSparseDatasetV3(PREPROCESSED_FILE, 'val')
gc.collect()

VAL_BATCH = min(BATCH_SIZE, len(val_ds))
effective = len(train_ds)
n_batches = effective // BATCH_SIZE
print(f'\nTrain: {n_batches} batches x {BATCH_SIZE:,} ({effective:,} eff)')
print(f'Gradient steps total: {n_batches * EPOCHS:,}')

---
## Treinar (60 epochs, cosine LR)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = torch.cuda.is_available()
model = PelancaNNUEv3().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
scaler = torch.amp.GradScaler('cuda') if use_amp else None
best_val = float('inf')
hist = {'train': [], 'val': [], 'lr': []}
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Treino: {EPOCHS} epochs | LR: {LR_MAX} -> {LR_MIN} | MSE puro')
print(f'Batches/epoch: {len(train_ds) // BATCH_SIZE}\n')
for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    tl, tn = 0.0, 0
    for stm_idx, stm_off, nstm_idx, nstm_off, ev in make_batches(train_ds, BATCH_SIZE, shuffle=True):
        stm_idx = stm_idx.to(device, non_blocking=True)
        stm_off = stm_off.to(device, non_blocking=True)
        nstm_idx = nstm_idx.to(device, non_blocking=True)
        nstm_off = nstm_off.to(device, non_blocking=True)
        ev = ev.to(device, non_blocking=True).unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.amp.autocast('cuda'):
                loss = nnue_loss(model(stm_idx, stm_off, nstm_idx, nstm_off), ev)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = nnue_loss(model(stm_idx, stm_off, nstm_idx, nstm_off), ev)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        tl += loss.item(); tn += 1
    train_loss = tl / max(tn, 1)
    model.eval()
    vl, vn = 0.0, 0
    with torch.no_grad():
        for stm_idx, stm_off, nstm_idx, nstm_off, ev in make_batches(val_ds, VAL_BATCH, shuffle=False):
            stm_idx = stm_idx.to(device)
            stm_off = stm_off.to(device)
            nstm_idx = nstm_idx.to(device)
            nstm_off = nstm_off.to(device)
            ev = ev.to(device).unsqueeze(1)
            loss = nnue_loss(model(stm_idx, stm_off, nstm_idx, nstm_off), ev)
            vl += loss.item(); vn += 1
    val_loss = vl / max(vn, 1)
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']
    hist['train'].append(train_loss)
    hist['val'].append(val_loss)
    hist['lr'].append(lr)
    mk = ''
    if val_loss < best_val:
        best_val = val_loss
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'val': val_loss},
                   os.path.join(CKPT_DIR, 'best.pt'))
        mk = ' ** BEST'
    elapsed = time.time() - t0
    print(f'E{epoch:3d} | t={train_loss:.6f} v={val_loss:.6f} lr={lr:.1e} | {elapsed:.0f}s{mk}')
    if (epoch + 1) % 15 == 0:
        p = os.path.join(CKPT_DIR, f'e{epoch}.nnue')
        export_nnue_v3(model, p, verbose=False)
        print(f'  -> {p}')
print(f'\nMelhor val: {best_val:.6f}')

---
## Grafico

In [ ]:
import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(hist['train'], label='Train'); a1.plot(hist['val'], label='Val')
a1.set_xlabel('Epoch'); a1.set_ylabel('Loss'); a1.legend(); a1.grid(alpha=0.3)
a1.set_title('Loss')
a2.plot(hist['lr']); a2.set_xlabel('Epoch'); a2.set_ylabel('LR')
a2.set_title('Learning Rate (Cosine)'); a2.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/kaggle/working/loss_v2.png', dpi=150); plt.show()

---
## Exportar Modelo Final

In [ ]:
ck = torch.load(os.path.join(CKPT_DIR, 'best.pt'), map_location='cpu')
fm = PelancaNNUEv3()
fm.load_state_dict(ck['model'])
print(f'Best: epoch {ck["epoch"]}, val={ck["val"]:.6f}\n')
export_nnue_v3(fm, NNUE_OUTPUT)
print(f'\nPRONTO: {NNUE_OUTPUT} ({os.path.getsize(NNUE_OUTPUT):,} bytes)')

---
## Teste

In [ ]:
fm.eval()
tests = [
    ('rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1', 'Inicial'),
    ('rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1', '1.e4'),
    ('8/8/8/8/8/8/4K3/R3k3 w - - 0 1', 'KR vs K'),
    ('8/8/8/8/8/8/4k3/r3K3 b - - 0 1', 'kr vs K'),
    ('r1bqkbnr/pppp1ppp/2n5/1B2p3/4P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3', 'Ruy Lopez'),
    ('rnbqkb1r/pppp1ppp/5n2/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3', 'Italian'),
]

print(f'{"Pos":<12} {"Raw":>8} {"tanh":>8} {"~cp":>8}')
print('-' * 45)
for fen, desc in tests:
    s, wi, bi = fen_to_sparse_fast(fen)
    if s == 1:
        wi, bi = bi, wi

    stm_idx = torch.tensor(wi, dtype=torch.long).unsqueeze(0).to('cpu')
    stm_off = torch.tensor([0], dtype=torch.long)
    nstm_idx = torch.tensor(bi, dtype=torch.long).unsqueeze(0).to('cpu')
    nstm_off = torch.tensor([0], dtype=torch.long)

    # Flatten for EmbeddingBag
    stm_idx = stm_idx.squeeze(0)
    nstm_idx = nstm_idx.squeeze(0)

    with torch.no_grad():
        r = fm(stm_idx, stm_off, nstm_idx, nstm_off).item()
    t = np.tanh(r)
    cp = np.arctanh(np.clip(t, -0.9999, 0.9999)) * EVAL_SCALE
    print(f'{desc:<12} {r:>+8.3f} {t:>+8.3f} {cp:>+8.0f}')